In [18]:
import sys
import os
import random
import torch
import numpy as np
from pathlib import Path
import wandb
import pandas as pd
import torchvision.transforms as transforms
from PIL import Image
import re

In [19]:
sys.path.append(os.path.abspath("../"))

from importlib import reload

import Segmentation.Postprocessing.segmentation_postprocessing
import Segmentation.SAM.sam_lora
import Segmentation.SAM.sam_lora_util
import Segmentation.Util.env_utils
import Segmentation.Util.dataset_util

import JunctionDetection.SkeletonizeDetect.segmentation_junction_detection

reload(Segmentation.Postprocessing.segmentation_postprocessing)
reload(Segmentation.SAM.sam_lora)
reload(Segmentation.SAM.sam_lora_util)
reload(Segmentation.Util.env_utils)
reload(Segmentation.Util.dataset_util)
reload(JunctionDetection.SkeletonizeDetect.segmentation_junction_detection)

loaded environment variables from: /local/scratch/jhehli/ForkSight/Segmentation/.env
loaded environment variables from: /local/scratch/jhehli/ForkSight/Segmentation/.env


<module 'JunctionDetection.SkeletonizeDetect.segmentation_junction_detection' from '/local/scratch/jhehli/ForkSight/JunctionDetection/SkeletonizeDetect/segmentation_junction_detection.py'>

In [20]:
Segmentation.Util.env_utils.load_segmentation_env()

SEED = Segmentation.Util.env_utils.load_as("SEED", int, 42)

RAW_DATA_DIR = os.getenv("RAW_DATA_DIR")
MODEL_CHECKPOINTS_DIR = os.getenv("MODEL_CHECKPOINTS_DIR")
DATASETS_DIR = os.getenv("DATASETS_DIR")

HIGHRES_IMG_PATCHES_DIR_NAME = os.getenv(
    "HIGHRES_IMG_PATCHES_DIR_NAME", "img_patches_1024")
HIGHRES_MASK_PATCHES_DIR_NAME = os.getenv(
    "HIGHRES_MASK_PATCHES_DIR_NAME", "mask_patches_1024")
HIGHRES_IMG_DIR_NAME = os.getenv("HIGHRES_IMG_DIR_NAME", "images_4096")
HIGHRES_MASK_DIR_NAME = os.getenv("HIGHRES_MASK_DIR_NAME", "masks_4096")

WANDB_ENTITY = os.getenv("WANDB_ENTITY", "EM_IMCR_BIOVSION")
WANDB_PROJECT = os.getenv("WANDB_PROJECT", "ForkSight-SAM")

DATASET_JUNCTION_COORDS_CVAT_XML_PATH = os.getenv(
    "DATASET_JUNCTION_COORDS_CVAT_XML_PATH", None)

if RAW_DATA_DIR is None or MODEL_CHECKPOINTS_DIR is None or DATASETS_DIR is None:
    raise ValueError(
        "RAW_DATA_DIR, MODEL_CHECKPOINTS_DIR or DATASETS_DIR environment variable not set.")

loaded environment variables from: /local/scratch/jhehli/ForkSight/Segmentation/.env


In [21]:
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

In [22]:
global DOWNSAMPLE_SIZE
DOWNSAMPLE_SIZE = None


def load_transform_image(path: Path, is_mask: bool = False):
    transform_steps = []

    if DOWNSAMPLE_SIZE is not None and not is_mask:
        transform_steps.append(transforms.Resize(
            DOWNSAMPLE_SIZE, interpolation=(transforms.InterpolationMode.BILINEAR)))

    transform_steps += [
        transforms.Resize((1024, 1024), interpolation=(
            transforms.InterpolationMode.NEAREST if is_mask else transforms.InterpolationMode.BILINEAR)),
        transforms.ToTensor(),
        transforms.Lambda(lambda t: t.repeat(3, 1, 1)
                          if t.shape[0] == 1 and not is_mask else t)
    ]

    transform = transforms.Compose(transform_steps)

    img = Image.open(path)
    return transform(img)


def get_single_image_input_list(image: torch.Tensor):
    return [{
        "image": img,
        "original_size": (img.shape[1], img.shape[2])
    } for img in image.unsqueeze(0).unbind(0)]

In [23]:
print("cuda available: ", torch.cuda.is_available())
!nvidia-smi --query-gpu=index,utilization.gpu --format=csv

device_idx = int(input("Enter the index of the cuda device: "))
device = torch.device(f"cuda:{device_idx}" if torch.cuda.is_available() else "cpu")

print(f"\nUsing device: {device}")

cuda available:  True
index, utilization.gpu [%]
0, 0 %
1, 0 %
2, 100 %
3, 100 %
4, 0 %
5, 100 %
6, 0 %
7, 0 %

Using device: cuda:1


In [ ]:
METRICS_DIR = Path(".")
METRICS_PATTERN = re.compile(r"^metrics_(\d{8}_\d{6})\.csv$")


def load_latest_metrics_csv() -> pd.DataFrame:
    candidates = []
    for p in METRICS_DIR.glob("metrics_*.csv"):
        m = METRICS_PATTERN.match(p.name)
        if m:
            candidates.append(p)

    if not candidates:
        return pd.DataFrame()

    latest = max(candidates, key=lambda p: p.stat().st_mtime)

    df_prev = pd.read_csv(latest)
    df_prev = df_prev.set_index("Model")
    df_prev.index = df_prev.index.astype(str)

    print(f"Loaded previous metrics from: {latest} (rows={len(df_prev)})")
    return df_prev


def compute_metrics(output_mask, groundtruth_mask):
    dice = Segmentation.SAM.sam_lora_util.hard_dice_score(
        output_mask, groundtruth_mask)
    iou = Segmentation.SAM.sam_lora_util.iou_score(
        output_mask, groundtruth_mask)
    output_mask_np = output_mask.squeeze(0).cpu().numpy()
    groundtruth_mask_np = groundtruth_mask.squeeze(0).cpu().numpy()
    clDice, tprec, tsens = Segmentation.SAM.sam_lora_util.hard_clDice(
        output_mask_np, groundtruth_mask_np)

    return dice, iou, clDice, tprec, tsens


params_artifact_suffix = "_params_minloss:v0"
loss_config_keys = ["bce_loss_weight", "cl_dice_loss_weight", "dice_loss_weight", "focal_loss_weight",
                    "junction_heatmap_weight_scale", "junction_patch_weight", "skeleton_recall_loss_weight", "topological_loss_weight"]

models = [
    "SAM_LoRA_Finetuning_20260212_155155",
    "SAM_LoRA_Finetuning_20260212_155412",
    "SAM_LoRA_Finetuning_20260212_155506",
    "SAM_LoRA_Finetuning_20260212_155528",
    "SAM_LoRA_Finetuning_20260212_155556",
    "SAM_LoRA_Finetuning_20260212_161635",
    "SAM_LoRA_Finetuning_20260218_112642-clDice",
    "SAM_LoRA_Finetuning_20260218_112642-Skel"
]

df_prev = load_latest_metrics_csv()
computed_models = set(df_prev.index) if not df_prev.empty else set()

results = {}

api = wandb.Api()
runs = runs = [run for run in list(wandb.Api().runs(
    f"{WANDB_ENTITY}/{WANDB_PROJECT}")) if Segmentation.SAM.sam_lora_util.EVALUATED_TAG in run.tags and run.state == "finished"
    and run.name in models]
runs = [run for run in runs if str(run.name) not in computed_models]

for run in runs:
    print(f"Evaluating run {run.name}")

    run_artifacts = [a for a in list(
        run.logged_artifacts()) if a.type == "model"]
    artifact = next(
        (a for a in run_artifacts if a.name.endswith(params_artifact_suffix)), None)
    if artifact is None:
        raise ValueError(
            f"No artifact of type 'model' found with name '{params_artifact_suffix}'")
    print(f"Found artifact: {artifact.name}")

    params, _ = Segmentation.SAM.sam_lora_util.get_params_from_artifact(
        artifact, device)
    model = Segmentation.SAM.sam_lora_util.initialize_sam_lora_with_params(
        run.config, params, device)
    model.eval()

    DOWNSAMPLE_SIZE = run.config.get("dataset_downsample_size", None)
    if DOWNSAMPLE_SIZE is not None:
        DOWNSAMPLE_SIZE = (DOWNSAMPLE_SIZE[0], DOWNSAMPLE_SIZE[1]) if isinstance(
            DOWNSAMPLE_SIZE, list) else DOWNSAMPLE_SIZE
    print(f"Downsample size: {DOWNSAMPLE_SIZE}")

    # Use same test dirs derived from the run's config
    model_test_img_dir = Path(
        DATASETS_DIR) / run.config["dataset"] / "test" / HIGHRES_IMG_PATCHES_DIR_NAME
    model_test_mask_dir = Path(
        DATASETS_DIR) / run.config["dataset"] / "test" / HIGHRES_MASK_PATCHES_DIR_NAME

    dice_scores, iou_scores, clDice_scores, tprec_scores, tsens_scores = [], [], [], [], []
    pp_dice_scores, pp_iou_scores, pp_clDice_scores, pp_tprec_scores, pp_tsens_scores = [], [], [], [], []

    for img_path in model_test_img_dir.glob("*.png"):
        groundtruth_mask_path = model_test_mask_dir / img_path.name

        img = load_transform_image(img_path)
        input_list = Segmentation.SAM.sam_lora_util.get_batched_input_list(
            img.unsqueeze(0).to(device))
        output = model(batched_input=input_list, multimask_output=False)

        output_mask = output[0]['masks'].detach().cpu()

        # "raw" model output
        groundtruth_mask = load_transform_image(
            groundtruth_mask_path, is_mask=True).detach().cpu()
        dice, iou, clDice, tprec, tsens = compute_metrics(
            output_mask, groundtruth_mask)
        dice_scores.append(dice)
        iou_scores.append(iou)
        clDice_scores.append(clDice)
        tprec_scores.append(tprec)
        tsens_scores.append(tsens)

        # post-processed (small objects removed) model output
        postprocessed_output_mask = Segmentation.Postprocessing.segmentation_postprocessing.remove_small_objects_from_batch(
            output_mask)
        pp_dice, pp_iou, pp_clDice, pp_tprec, pp_tsens = compute_metrics(
            postprocessed_output_mask, groundtruth_mask)
        pp_dice_scores.append(pp_dice)
        pp_iou_scores.append(pp_iou)
        pp_clDice_scores.append(pp_clDice)
        pp_tprec_scores.append(pp_tprec)
        pp_tsens_scores.append(pp_tsens)

    results[run.name] = {
        "dataset": run.config["dataset"],

        "Dice": np.mean(dice_scores),
        "IoU": np.mean(iou_scores),
        "clDice": np.mean(clDice_scores),
        "tprec": np.mean(tprec_scores),
        "tsens": np.mean(tsens_scores),

        "Dice Postprocessed": np.mean(pp_dice_scores),
        "IoU Postprocessed": np.mean(pp_iou_scores),
        "clDice Postprocessed": np.mean(pp_clDice_scores),
        "tprec Postprocessed": np.mean(pp_tprec_scores),
        "tsens Postprocessed": np.mean(pp_tsens_scores),
    }

    for k in loss_config_keys:
        results[run.name][k] = run.config.get(k, 0.0)

    del model, params
    torch.cuda.empty_cache()

Loaded previous metrics from: metrics_20260219_154059.csv (rows=9)
Evaluating run SAM_LoRA_Finetuning_20260219_150640
Found artifact: sam_lora_finetuning_20260219_150640_briwbyzb_params_minloss:v0


wandb:   1 of 1 files downloaded.  


Downsample size: (256, 256)


In [25]:
df = pd.DataFrame(results).T
df.index.name = "Model"

if not df_prev.empty:
    df = pd.concat([df_prev, df], axis=0)
    df = df[~df.index.duplicated(keep="first")]

best_per_metric = df.idxmax()

metric_cols = ["Dice", "IoU", "clDice", "tprec", "tsens",
               "Dice Postprocessed", "IoU Postprocessed", "clDice Postprocessed", "tprec Postprocessed", "tsens Postprocessed"]


def highlight_best(s):
    if s.name not in metric_cols:
        return ['' for _ in s]
    is_best = s == s.max()
    return ['font-weight: bold' if v else '' for v in is_best]


df.style.apply(highlight_best, axis=0).format(
    {col: "{:.4f}" for col in metric_cols}
)

,dataset,Dice,IoU,clDice,tprec,tsens,Dice Postprocessed,IoU Postprocessed,clDice Postprocessed,tprec Postprocessed,tsens Postprocessed,bce_loss_weight,cl_dice_loss_weight,dice_loss_weight,focal_loss_weight,junction_heatmap_weight_scale,junction_patch_weight,skeleton_recall_loss_weight,topological_loss_weight
Model,,,,,,,,,,,,,,,,,,,
SAM_LoRA_Finetuning_20260212_155155,SAM_LoRA_Augmented_Final_SoI,0.8028,0.6724,0.9348,0.9733,0.9026,0.8006,0.6695,0.9316,0.9804,0.8914,0.500000,0.000000,0.000000,0.000000,0,0,0.000000,0.500000
SAM_LoRA_Finetuning_20260212_155412,SAM_LoRA_Augmented_Final_SoI,0.8226,0.7002,0.9497,0.9754,0.9274,0.8214,0.6987,0.9477,0.9832,0.9174,0.500000,0.500000,0.000000,0.000000,0,0,0.000000,0.500000
SAM_LoRA_Finetuning_20260212_155506,SAM_LoRA_Augmented_Final_SoI,0.8377,0.7224,0.9564,0.9730,0.9420,0.8385,0.7238,0.9567,0.9877,0.9298,0.250000,0.500000,0.500000,0.000000,0,0,0.000000,0.000000
SAM_LoRA_Finetuning_20260212_155528,SAM_LoRA_Augmented_Final_SoI,0.8452,0.7333,0.9456,0.9299,0.9639,0.8531,0.7452,0.9594,0.9640,0.9565,0.250000,0.000000,0.500000,0.000000,0,0,0.500000,0.000000
SAM_LoRA_Finetuning_20260212_155556,SAM_LoRA_Augmented_Final_SoI,0.8329,0.7153,0.9470,0.9508,0.9459,0.8369,0.7214,0.9516,0.9738,0.9334,0.250000,0.500000,0.500000,0.000000,1000,0,0.000000,0.000000
SAM_LoRA_Finetuning_20260212_161635,SAM_LoRA_Augmented_Final_SoI,0.8355,0.7191,0.9550,0.9734,0.9392,0.8355,0.7194,0.9547,0.9878,0.9262,0.000000,0.500000,0.500000,0.250000,0,0,0.000000,0.000000
SAM_LoRA_Finetuning_20260218_112642-Skel,SAM_LoRA_Augmented_Final_SoI_Oversampled,0.8442,0.7319,0.9430,0.9200,0.9690,0.8540,0.7467,0.9605,0.9603,0.9621,0.250000,0.000000,0.500000,0.000000,0,0,0.500000,0.000000
SAM_LoRA_Finetuning_20260218_112642-clDice,SAM_LoRA_Augmented_Final_SoI_Oversampled,0.8416,0.7281,0.9587,0.9725,0.9466,0.8430,0.7304,0.9598,0.9872,0.9358,0.250000,0.500000,0.500000,0.000000,0,0,0.000000,0.000000
SAM_LoRA_Finetuning_20260219_093322,SAM_LoRA_Augmented_Final_SoI_Oversampled,0.8413,0.7275,0.9589,0.9712,0.9483,0.8431,0.7303,0.9600,0.9847,0.9384,0.000000,0.750000,0.250000,0.000000,0,0,0.000000,0.000000


In [26]:
from datetime import datetime
df.to_csv(f'metrics_{datetime.now().strftime("%Y%m%d_%H%M%S")}.csv')